In [18]:
import pandas as pd
import os

# Cargar datos
ruta = os.path.join('..', 'Data', 'TouristAccommodationRaw19012026.csv')

df = pd.read_csv(ruta)

## 1. Limpieza y Preparación: Marketing y Estrategia Comercial

En esta fase, preparamos los datos para el análisis de mercado. El objetivo principal es normalizar las categorías y asegurar que la métrica de **precio** sea íntegra sin reducir el tamaño de la muestra (7001 registros).

* **Normalización de Texto:** Se eliminan espacios en blanco en las columnas `city` y `room_type` para evitar duplicidad de categorías.
* **Imputación de Precios:** Siguiendo la recomendación de no eliminar registros críticos, se aplica una **imputación por segmentos**.
    * **Lógica:** Se utiliza la **mediana** calculada por la combinación de ciudad y tipo de alojamiento. Esto garantiza que el precio asignado sea realista según su contexto geográfico y tipo de propiedad.
    * **Respaldo:** Se aplica una mediana global para cubrir cualquier caso excepcional sin datos segmentados.
* **Consistencia de Tipos:** Se fuerza el tipado a `float` para permitir cálculos estadísticos precisos.

In [19]:
# --- 1. NORMALIZACIÓN DE CATEGORÍAS ---
df['city'] = df['city'].str.strip()
df['room_type'] = df['room_type'].str.strip()

# --- 2. IMPUTACIÓN DE PRECIOS ---

# Paso A: Imputación segmentada (Por Ciudad y Tipo de Habitación)
df['price'] = df.groupby(['city', 'room_type'])['price'].transform(
    lambda x: x.fillna(x.median())
)

# Paso B: Imputación de seguridad
# En caso de que un segmento completo sea nulo, usamos la mediana de todo el dataset
df['price'] = df['price'].fillna(df['price'].median())

# --- 3. AJUSTE DE TIPOS ---
# Aseguramos que price sea numérico para el análisis comercial
df['price'] = df['price'].astype(float)


## 2. Limpieza para Experiencia del Cliente.

Para responder a las preguntas sobre satisfacción sin reducir la muestra total de 7001 registros, se aplica la siguiente lógica:

* **Tratamiento de Ratings:** Se transforman a formato numérico, manteniendo los valores ausentes como `NaN`. Esto permite que el cálculo de la **puntuación media** sea real, ya que las funciones estadísticas de Python ignoran los nulos en lugar de promediarlos como ceros.
* **Indicador de Calidad:** Se crea la columna `high_quality_stay`. Para el cálculo del porcentaje por ciudad, se recomendará al analista basarse solo en el universo de alojamientos con puntuación existente para no sesgar el resultado hacia abajo.
* **Métricas de Actividad:** Se imputan con `0` los valores nulos en `reviews_per_month`, asumiendo que la ausencia de dato indica falta de actividad reciente.

In [20]:
# --- LIMPIEZA PARA EXPERIENCIA DEL CLIENTE ---

# 1. Convertir a numérico. Los errores o celdas vacías se convierten en NaN.
df['review_scores_rating'] = pd.to_numeric(df['review_scores_rating'], errors='coerce')

# 2. Crear el indicador para la pregunta de negocio (> 80)
# Esta columna será True para los excelentes, False para los de 80 o menos, 
# y NaN para los que no tienen evaluación.
df['high_quality_stay'] = df['review_scores_rating'] > 80

# 3. Limpieza de reviews por mes (opcional pero recomendada)
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)


df["insert_date"] = pd.to_datetime(df["insert_date"], errors="coerce")  # convierte y pone NaT en errores
df = df.dropna(subset=["insert_date"])  # elimina filas con fechas inválidas
df["month"] = df["insert_date"].dt.to_period("M")

C:\Users\luisc\AppData\Local\Temp\ipykernel_24500\2467235495.py:15: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["insert_date"] = pd.to_datetime(df["insert_date"], errors="coerce")  # convierte y pone NaT en errores


## 3. Normalización Temporal y Validación de Disponibilidad

* **Procesamiento de Fechas:** * Conversión de `insert_date` a formato `datetime` mediante coerción de errores.
    * Generación de la columna `month` (Periodo) para análisis de series temporales.
* **Estandarización de Disponibilidad:** * Mapeo de valores `"VERDADERO"` y `"NULL"` a lógica booleana.
    * Conversión final de la columna `has_availability` a tipo `bool` para optimizar el filtrado y el almacenamiento.

In [21]:
# --- 1. NORMALIZACIÓN TEMPORAL ---
# Conversión a datetime: 'errors="coerce"' transforma formatos inválidos en NaT
df["insert_date"] = pd.to_datetime(df["insert_date"], errors="coerce")

# Extracción del periodo mensual para facilitar agregaciones temporales
df["month"] = df["insert_date"].dt.to_period("M")

# --- 2. VALIDACIÓN DE DISPONIBILIDAD ---
# Homogeneización de valores categóricos a booleanos y manejo de nulos
df["has_availability"] = df["has_availability"].replace({
    "VERDADERO": True, 
    "NULL": False
}).fillna(False).astype(bool)

C:\Users\luisc\AppData\Local\Temp\ipykernel_24500\1242647963.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  }).fillna(False).astype(bool)


## 4. Exportación de Resultados

Una vez finalizado el proceso de limpieza y normalización para los tres departamentos se procede a exportar el **Dataset Limpio**.

* **Ruta de destino:** Se almacena en la carpeta institucional `/Data/` bajo el nombre `TouristAccommodationClean19012026.csv`.
* **Codificación:** Se utiliza `utf-8-sig` para garantizar que la corrección de caracteres especiales.

In [22]:
# --- EXPORTACIÓN DEL DATASET LIMPIO ---

# 1. Nombre del nuevo archivo
nombre_archivo_limpio = 'TouristAccommodationClean19012026.csv'

# 2. Construimos la ruta apuntando a la misma carpeta 'Data'
# Usamos '..' para subir un nivel y luego entrar en 'Data'
ruta_guardado = os.path.join('..', 'Data', nombre_archivo_limpio)

# 3. Guardamos el DataFrame
# index=False evita que se cree una columna extra de números
# encoding='utf-8-sig' 
df.to_csv(ruta_guardado, index=False, encoding='utf-8-sig')
